# NEISS-AI quickstart: US e-bike and e-scooter injuries, 2019–2025

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subash28kumar/neiss-ai/blob/main/notebooks/NEISS-AI_Quickstart.ipynb)

This notebook reproduces the main numbers on the [NEISS-AI website](https://subash28kumar.github.io/neiss-ai/) in a few minutes, with no setup on your computer.

**What you will get**

1. National estimates of emergency department (ED) visits for e-bikes, e-scooters, bicycles and other devices, every year from 2019 to 2025, with 95% confidence intervals.
2. A trend chart.
3. The share of injured riders who were admitted to the hospital, e-bikes compared with bicycles.

**Where the data comes from.** The US Consumer Product Safety Commission (CPSC) runs NEISS, a national sample of about 100 hospital emergency departments. CPSC publishes one public file per year. NEISS-AI reads the short injury story in each record so that e-bikes are counted the same way in every year, including before 2024, when e-bikes shared a product code with mopeds.

**How to run it.** Click **Runtime → Run all**. The whole notebook takes about 3–5 minutes, mostly for downloading the data.

## Step 1. Install NEISS-AI

NEISS-AI is a free, open-source Python package ([PyPI](https://pypi.org/project/neissai/), [GitHub](https://github.com/subash28kumar/neiss-ai)).

In [ ]:
%pip install -q neissai

import neissai
import pandas as pd
print("NEISS-AI version", neissai.__version__)

## Step 2. Get the NEISS data files

This step downloads the public NEISS file for each year from CPSC into a folder called `data`. Each file is about 30–60 MB.

If a download fails (CPSC sometimes changes its file locations or blocks automated downloads), the cell tells you what to do. In short: download the yearly files by hand from the [CPSC NEISS page](https://www.cpsc.gov/Research--Statistics/NEISS-Injury-Data) (for example `neiss2025.tsv`), then upload them when asked.

In [ ]:
from pathlib import Path

YEARS = list(range(2019, 2026))      # 2019 to 2025
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def have(year):
    return any(str(year) in p.name and "neiss" in p.name.lower() for p in DATA_DIR.iterdir())

for year in YEARS:
    if have(year):
        print(year, "already here")
        continue
    try:
        neissai.download([year], DATA_DIR)
        print(year, "downloaded")
    except Exception as err:
        print(year, "could not be downloaded automatically:", err)

missing = [y for y in YEARS if not have(y)]
if missing:
    print("\nStill missing:", missing)
    print("Download these years from https://www.cpsc.gov/Research--Statistics/NEISS-Injury-Data")
    try:
        from google.colab import files     # only works inside Google Colab
        print("Then choose the files in the upload box below.")
        uploaded = files.upload()
        for name, content in uploaded.items():
            (DATA_DIR / name).write_bytes(content)
        missing = [y for y in YEARS if not have(y)]
    except ImportError:
        print("Outside Colab: copy the files into the 'data' folder, then run this cell again.")
print("\nReady." if not missing else f"\nStill missing {missing}; the next steps need every year.")

## Step 3. Read the files and label each injury

`label_devices` decides which device each injury involved (e-bike, e-scooter, bicycle, kick scooter, hoverboard, skateboard or moped). It uses CPSC's product codes and key words first. For older records where the code mixes e-bikes and mopeds and the story is unclear, a small text classifier reads the story and decides. `add_outcomes` adds simple flags such as "admitted to the hospital".

This takes about a minute.

In [ ]:
df = neissai.read_neiss(DATA_DIR, years=f"{YEARS[0]}-{YEARS[-1]}")
df = neissai.add_outcomes(neissai.label_devices(df))
micromobility = df["device"].isin(neissai.DEVICES)

print(f"{len(df):,} NEISS records read; {micromobility.sum():,} involve a micromobility device.")
df.loc[micromobility, "device"].value_counts().rename("records in the sample")

## Step 4. National estimates by year and device

NEISS is a sample, so each record has a weight that says how many national ED visits it stands for. `national_estimates` adds up the weights and computes a 95% confidence interval that respects the sample design. Note that I pass the **whole** table plus a filter (`where=`), not a filtered table, so the uncertainty uses every hospital in the sample.

Numbers are rounded to the nearest 100, as on the website. An asterisk (*) marks an estimate CPSC would call potentially unstable (fewer than 20 records, under 1,200, or a coefficient of variation above 33%).

In [ ]:
est = neissai.national_estimates(df, by=["year", "device"], where=micromobility)

def r100(x):
    return int(x / 100 + 0.5) * 100

est["shown"] = [f"{r100(e):,}" + ("*" if u else "") for e, u in zip(est["estimate"], est["unstable"])]
order = ["E-bike", "E-scooter", "Kick scooter", "Bicycle", "Hoverboard", "Skateboard", "Moped / minibike"]
table = est.pivot(index="device", columns="year", values="shown").reindex(order)
table

### The headline number

In [ ]:
eb = est[(est["device"] == "E-bike")].set_index("year")
y = YEARS[-1]
row = eb.loc[y]
change = row["estimate"] / eb.loc[y - 1, "estimate"] - 1
print(f"E-bike ED visits in {y}: {r100(row['estimate']):,} "
      f"(95% CI {r100(row['ci_low']):,} to {r100(row['ci_high']):,}), "
      f"up {change:.0%} from {y - 1} and {row['estimate'] / eb.loc[2022, 'estimate']:.1f} times the 2022 level.")

## Step 5. Chart: newer devices over time

The shaded bands are 95% confidence intervals. Hollow points are potentially unstable estimates. Bicycles (about 400,000 a year) are left out so the newer devices are easy to see.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
for device, color in [("E-scooter", "#d9622b"), ("Kick scooter", "#1a9e77"), ("E-bike", "#2c6fbb")]:
    d = est[est["device"] == device].sort_values("year")
    ax.fill_between(d["year"], d["ci_low"], d["ci_high"], color=color, alpha=0.15)
    ax.plot(d["year"], d["estimate"], color=color, lw=2, label=device)
    ax.scatter(d["year"], d["estimate"], color=[("white" if u else color) for u in d["unstable"]],
               edgecolor=color, zorder=3)
ax.set_title("US emergency department visits by device, 2019–2025")
ax.set_ylabel("Estimated visits per year")
ax.yaxis.set_major_formatter(lambda v, _: f"{v / 1000:.0f}k")
ax.set_ylim(bottom=0)
ax.legend(frameon=False)
ax.text(0, -0.16, "Source: NEISS-AI v1.4.0 (Kumar 2026), national estimates from CPSC NEISS. "
        "https://doi.org/10.5281/zenodo.23550272", transform=ax.transAxes, fontsize=7, color="gray")
plt.tight_layout()
plt.show()

## Step 6. How often does an injury mean a hospital stay?

`shares` gives the weighted share of injured riders who were admitted, transferred or held in the hospital, with a 95% confidence interval. I pool 2024 and 2025, the two years with CPSC's separate e-bike code.

In [ ]:
recent = micromobility & df["year"].isin([2024, 2025])
hosp = neissai.shares(df, "hospitalized", by=["device"], where=recent)
hosp = hosp[hosp["device"].isin(["E-bike", "E-scooter", "Bicycle"])].copy()
for c in ["share", "ci_low", "ci_high"]:
    hosp[c] = (hosp[c] * 100).round(1)
hosp[["device", "share", "ci_low", "ci_high", "records"]].rename(
    columns={"share": "% hospitalized", "ci_low": "95% CI low", "ci_high": "95% CI high"})

## Step 7. Save your results

This saves the full table (estimate, 95% CI, coefficient of variation, records and the unstable flag) as a CSV file. In Colab, it also downloads the file to your computer.

In [ ]:
out = est.drop(columns="shown")
out.to_csv("neissai_estimates_by_year_device.csv", index=False)
print("Saved neissai_estimates_by_year_device.csv")
try:
    from google.colab import files
    files.download("neissai_estimates_by_year_device.csv")
except ImportError:
    pass

## Limits to keep in mind

- NEISS counts **emergency department visits**, not riders or miles. These are counts, not risk rates.
- Injuries treated elsewhere (urgent care, doctors' offices) and deaths at the scene are not included.
- Early e-bike estimates (2019–2020) rest on small samples and are marked as potentially unstable.
- The powered-scooter code may include some seated, moped-style scooters.

## Go further

- Try other groupings, for example `by=["year", "device", "age_group"]` or `by=["year", "device", "Sex"]`.
- Search the injury stories: `df[df["Narrative"].str.contains("BATTERY", case=False)]`.
- The full website: https://subash28kumar.github.io/neiss-ai/

## Cite this tool

If you use NEISS-AI or these numbers, please cite:

> Kumar S. NEISS-AI: AI-assisted national injury estimates from NEISS emergency department narratives. Version 1.4.0 [software]. Zenodo; 2026. https://doi.org/10.5281/zenodo.23550272

Data come from the US Consumer Product Safety Commission. CPSC does not endorse this tool. Questions or ideas: open an issue at https://github.com/subash28kumar/neiss-ai/issues.

In [ ]:
print("Kumar S. NEISS-AI: AI-assisted national injury estimates from NEISS emergency department "
      f"narratives. Version {neissai.__version__} [software]. Zenodo; 2026. https://doi.org/10.5281/zenodo.23550272")